# Эксперимент: множитель количества до округления и пороги по горизонту

Поверх v5 (тренд категории, глубина 6, вес свежих недель 26, целое под SMAPE, порог 0.5, ансамбль из двух сидов).

- `amount_scale` - множитель количества регрессора **до** выбора целого. Нули он не меняет (SMAPE на неделях без продаж тот же), а объём на неделях с продажей - да. Модель занижает объём: прогноз / факт ~ 0.78-0.85 на бэктесте. Прежний `scale` в `make_forecast` применялся после округления и снова давал дробь - это другое.
- Порог парой (недели 1-4, недели 5-9): к дальним неделям вероятности размытее, один порог на все недели может быть неоптимален. Только две группы - иначе подгонка под фолды.

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_scale"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; первый - v5, с ним и сравниваем.

In [2]:
VARIANTS = {"v5": V5}
for s in (1.1, 1.2, 1.3):
    for t in (0.5, 0.55):
        VARIANTS[f"множитель {s}, порог {t}"] = {**V5, "amount_scale": s, "threshold": t}
for near, far in ((0.45, 0.5), (0.5, 0.45), (0.5, 0.55), (0.55, 0.5), (0.45, 0.55), (0.55, 0.45)):
    VARIANTS[f"порог {near} / {far}"] = {**V5, "threshold": (near, far)}
for s in (1.1, 1.2):
    VARIANTS[f"множитель {s}, порог 0.5 / 0.55"] = {**V5, "amount_scale": s, "threshold": (0.5, 0.55)}
list(VARIANTS)

['v5',
 'множитель 1.1, порог 0.5',
 'множитель 1.1, порог 0.55',
 'множитель 1.2, порог 0.5',
 'множитель 1.2, порог 0.55',
 'множитель 1.3, порог 0.5',
 'множитель 1.3, порог 0.55',
 'порог 0.45 / 0.5',
 'порог 0.5 / 0.45',
 'порог 0.5 / 0.55',
 'порог 0.55 / 0.5',
 'порог 0.45 / 0.55',
 'порог 0.55 / 0.45',
 'множитель 1.1, порог 0.5 / 0.55',
 'множитель 1.2, порог 0.5 / 0.55']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

season2025 готов


season2024 готов


recent готов


bias                         rmse  \
фолд                            recent season2024 season2025 recent   
вариант                                                               
v5                               76.53      86.25      81.10   3.34   
множитель 1.1, порог 0.5         83.63      90.87      85.00   3.22   
множитель 1.1, порог 0.5 / 0.55  81.44      89.71      83.02   3.22   
множитель 1.1, порог 0.55        82.04      89.99      83.89   3.22   
множитель 1.2, порог 0.5         91.88      94.75      88.25   3.24   
множитель 1.2, порог 0.5 / 0.55  87.27      92.63      85.06   3.24   
множитель 1.2, порог 0.55        87.25      92.72      86.00   3.24   
множитель 1.3, порог 0.5         97.31      98.87      91.24   3.39   
множитель 1.3, порог 0.55        91.10      95.88      88.65   3.38   
порог 0.45 / 0.5                 75.79      85.96      79.75   3.34   
порог 0.45 / 0.55                73.80      84.92      78.75   3.34   
порог 0.5 / 0.45                 75.47      86.09      80.32   3.34   
порог 0.5 / 0.55                 75.54      86.00      80.40   3.34   
порог 0.55 / 0.45                73.62      84.65      79.30   3.34   
порог 0.55 / 0.5                 74.99      85.22      80.84   3.34   

                                                       smape             \
фолд                            season2024 season2025 recent season2024   
вариант                                                                   
v5                                    1.93       2.65  58.74      42.81   
множитель 1.1, порог 0.5              1.88       2.60  58.78      42.76   
множитель 1.1, порог 0.5 / 0.55       1.88       2.60  58.52      42.71   
множитель 1.1, порог 0.55             1.88       2.60  58.25      42.67   
множитель 1.2, порог 0.5              1.87       2.68  59.22      42.86   
множитель 1.2, порог 0.5 / 0.55       1.87       2.68  58.88      42.80   
множитель 1.2, порог 0.55             1.87       2.68  58.53      42.74   
множитель 1.3, порог 0.5              1.89       2.88  59.82      43.07   
множитель 1.3, порог 0.55             1.89       2.88  59.03      42.91   
порог 0.45 / 0.5                      1.93       2.64  59.23      43.01   
порог 0.45 / 0.55                     1.93       2.65  58.98      43.00   
порог 0.5 / 0.45                      1.93       2.64  59.35      43.17   
порог 0.5 / 0.55                      1.93       2.65  58.49      42.80   
порог 0.55 / 0.45                     1.93       2.64  59.10      43.16   
порог 0.55 / 0.5                      1.93       2.65  58.50      42.79   

                                             wape                        
фолд                            season2025 recent season2024 season2025  
вариант                                                                  
v5                                   57.58  47.07      62.87      51.20  
множитель 1.1, порог 0.5             57.49  47.56      63.76      51.86  
множитель 1.1, порог 0.5 / 0.55      57.51  47.45      63.54      51.73  
множитель 1.1, порог 0.55            57.46  47.34      63.45      51.62  
множитель 1.2, порог 0.5             57.64  49.33      65.26      53.47  
множитель 1.2, порог 0.5 / 0.55      57.61  49.00      64.87      53.20  
множитель 1.2, порог 0.55            57.54  48.73      64.63      52.97  
множитель 1.3, порог 0.5             57.99  51.84      67.34      56.23  
множитель 1.3, порог 0.55            57.86  51.02      66.48      55.64  
порог 0.45 / 0.5                     57.95  47.15      62.95      51.27  
порог 0.45 / 0.55                    58.00  47.12      62.93      51.29  
порог 0.5 / 0.45                     57.88  47.16      63.10      51.24  
порог 0.5 / 0.55                     57.63  47.05      62.85      51.23  
порог 0.55 / 0.45                    57.86  47.13      63.13      51.24  
порог 0.55 / 0.5                     57.56  47.04      62.90      51.21

## Выбор

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["v5"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["v5", "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print("v5 в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 в этом запуске: {'season2025': 0.8095, 'season2024': 0.8231, 'recent': 0.7869} | по сезону 0.8163
лучший вариант: порог 0.55 / 0.5 {'season2025': 0.8094, 'season2024': 0.8226, 'recent': 0.7841, 'по сезону': 0.816}


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
порог 0.55 / 0.5,0.8094,0.8226,0.7841,0.8160,True
v5,0.8095,0.8231,0.7869,0.8163,False
порог 0.5 / 0.55,0.8097,0.8232,0.7846,0.8164,False
порог 0.45 / 0.55,0.8109,0.8242,0.7866,0.8175,False
порог 0.45 / 0.5,0.8109,0.8248,0.7897,0.8179,False
"множитель 1.1, порог 0.5 / 0.55",0.8111,0.8247,0.7878,0.8179,False
порог 0.55 / 0.45,0.8099,0.8261,0.7871,0.8180,False
"множитель 1.1, порог 0.55",0.8114,0.8245,0.7862,0.8180,False
порог 0.5 / 0.45,0.8106,0.8269,0.7902,0.8188,False
